In [2]:
from text_review_01.document_processor import DocumentProcessor

In [4]:
from typing import Optional
import requests
import chardet


class TxtReader:
    """Reader for text files from local paths or URLs.
    
    Attributes:
        source (str): Path to local file or URL.
        encoding (Optional[str]): Text encoding. Auto-detected if None.
    """
    
    def __init__(self, source: str, encoding: Optional[str] = None) -> None:
        """Initialize the text reader.
        
        Args:
            source: Local file path or URL to text file.
            encoding: Text encoding. If None, will auto-detect.
        """
        self.source = source
        self.encoding = encoding
        
    def read(self) -> str:
        """Read text content from file or URL.
        
        Returns:
            Decoded text content.
            
        Raises:
            FileNotFoundError: If local file doesn't exist.
            requests.exceptions.RequestException: If URL request fails.
            UnicodeDecodeError: If encoding detection fails.
        """
        content = self._load_content()
        
        if self.encoding is None:
            self.encoding = self._detect_encoding(content)
            
        return content.decode(self.encoding)
    
    def _load_content(self) -> bytes:
        """Load raw bytes from local file or URL.
        
        Returns:
            Raw file content.
        """
        if self.source.startswith(('http://', 'https://')):
            return self._load_from_url()
        else:
            return self._load_from_file()
    
    def _load_from_file(self) -> bytes:
        """Read local file as bytes.
        
        Returns:
            File content.
            
        Raises:
            FileNotFoundError: If file doesn't exist.
        """
        with open(self.source, 'rb') as file:
            return file.read()
    
    def _load_from_url(self) -> bytes:
        """Fetch content from URL as bytes.
        
        Returns:
            URL content.
            
        Raises:
            requests.exceptions.RequestException: If request fails.
        """
        response = requests.get(self.source)
        response.raise_for_status()
        return response.content
    
    def _detect_encoding(self, content: bytes) -> str:
        """Auto-detect encoding if not provided.
        
        Args:
            content: Raw content to analyze.
            
        Returns:
            Detected encoding.
        """
        result = chardet.detect(content)
        return result['encoding'] or 'utf-8'


In [7]:
kjv_text = TxtReader(source=r"https://openbible.com/textfiles/kjv.txt")

In [13]:
t1 = kjv_text.read()
print(t1[:1000])

KJV
King James Bible: Pure Cambridge Edition - Text courtesy of www.BibleProtector.com
Genesis 1:1	In the beginning God created the heaven and the earth.
Genesis 1:2	And the earth was without form, and void; and darkness [was] upon the face of the deep. And the Spirit of God moved upon the face of the waters.
Genesis 1:3	And God said, Let there be light: and there was light.
Genesis 1:4	And God saw the light, that [it was] good: and God divided the light from the darkness.
Genesis 1:5	And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.
Genesis 1:6	And God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.
Genesis 1:7	And God made the firmament, and divided the waters which [were] under the firmament from the waters which [were] above the firmament: and it was so.
Genesis 1:8	And God called the firmament Heaven. And the evening and the morning were the second day.
Genesi

In [9]:
import requests
response = requests.get('https://openbible.com/textfiles/kjv.txt')
print(f"Status Code: {response.status_code}")
print(f"Content Length: {len(response.content)}")

Status Code: 200
Content Length: 4606957


In [11]:
reader = TxtReader('https://openbible.com/textfiles/kjv.txt')
try:
    bible_text = reader.read()
    print(f"Success! Got {len(bible_text)} characters")
    print(f"First 100 chars: {bible_text[:200]}")
except Exception as e:
    print(f"Error: {e}")

Success! Got 4602958 characters
First 100 chars: KJV
King James Bible: Pure Cambridge Edition - Text courtesy of www.BibleProtector.com
Genesis 1:1	In the beginning God created the heaven and the earth.
Genesis 1:2	And the earth was without form, an
